# EGFR mutation–TKI resistance pipeline

This notebook does two things:

1. Extracts experimental drug-response outcomes from the two Hayes Excel workbooks.
2. Trains a machine-learning model after you provide real mutation-level structural features.

**Important:** The Hayes files contain drug-response measurements, but they do not contain 3D structural features. Do not invent structural values. First extract the mutation list, then calculate or obtain those features.

## Step 1 — Upload the files

Run the cell below, click **Choose Files**, and select:

- `v2_Hayes Source Data Main.xlsx`
- `v2_Hayes Source Data Supplement.xlsx`
- `egfr_resistance_pipeline.py`

You may also upload `structural_features_template.csv`, but it is not required for the first extraction step.

In [ ]:
from google.colab import files

uploaded = files.upload()
print("\nUploaded files:")
for name in uploaded:
    print(" -", name)

## Step 2 — Install the required Python packages

In [ ]:
!pip -q install pandas openpyxl scikit-learn numpy

## Step 3 — Check that Colab can see the files

In [ ]:
import os

for name in sorted(os.listdir(".")):
    if name.endswith((".xlsx", ".py", ".csv")):
        print(name)

## Step 4 — Extract the experimental mutation–drug outcomes

This parses the normalized dose-response curves and calculates a viability AUC for each mutation–drug pair. Resistance is initially defined as:

`mutant AUC / EGFR-WT AUC ≥ 1.50`

The threshold can be changed later.

In [ ]:
from egfr_resistance_pipeline import load_hayes_workbooks, summarize_assays

main_file = "v2_Hayes Source Data Main.xlsx"
supp_file = "v2_Hayes Source Data Supplement.xlsx"

dose_response = load_hayes_workbooks([main_file, supp_file])
assay_outcomes = summarize_assays(dose_response, resistance_ratio=1.50)

print("Individual dose-response measurements:", len(dose_response))
print("Mutation-drug outcome rows:", len(assay_outcomes))
print("Distinct mutations:", assay_outcomes["mutation"].nunique())
print("Drugs:", sorted(assay_outcomes["drug"].unique()))

display(assay_outcomes.head(10))

## Step 5 — Save and download the extracted assay results

In [ ]:
dose_response.to_csv("dose_response_long.csv", index=False)
assay_outcomes.to_csv("assay_outcomes.csv", index=False)

from google.colab import files
files.download("assay_outcomes.csv")

## Step 6 — Generate the complete structural-feature worksheet

The next cell creates one row for every mutation found in the Hayes data. Download it and fill the numeric columns using actual WT-versus-mutant structural calculations.

Keep the `mutation` values unchanged.

In [ ]:
import pandas as pd

feature_columns = [
    "mutation",
    "pocket_volume_change_pct",
    "hinge_distance_change_A",
    "contacts_lost",
    "contacts_gained",
    "binding_site_rmsd_A",
    "drug_distance_change_A",
    "delta_sasa_A2",
    "delta_stability_kcal_mol",
]

mutations = sorted(assay_outcomes["mutation"].dropna().unique())
structural_template = pd.DataFrame({"mutation": mutations})

for col in feature_columns[1:]:
    structural_template[col] = pd.NA

structural_template.to_csv("structural_features.csv", index=False)
display(structural_template)

files.download("structural_features.csv")

## Step 7 — Fill `structural_features.csv` outside Colab

Open the downloaded CSV in Excel or Google Sheets.

Each row represents one mutation. Fill the structural columns with real calculated values. Examples of valid inputs are pocket-volume change, contacts lost, binding-site RMSD, and drug-distance change.

Blank cells are allowed, but each structural feature needs enough measured mutations to be informative.

When finished:

1. Save the file as `structural_features.csv`.
2. Return to this notebook.
3. Run the upload cell below and select the completed file.

In [ ]:
from google.colab import files

completed_features = files.upload()
print("Uploaded:", list(completed_features))

## Step 8 — Verify that the structural file is usable

In [ ]:
structural_features = pd.read_csv("structural_features.csv")

print("Rows:", len(structural_features))
print("Mutations with at least one structural value:",
      structural_features.drop(columns="mutation").notna().any(axis=1).sum())

display(structural_features.head(10))

## Step 9 — Train and evaluate the machine-learning model

The model uses leave-one-mutation-out validation. Therefore, all measurements for a held-out mutation stay out of training during that test fold.

In [ ]:
!python egfr_resistance_pipeline.py \
  --main "v2_Hayes Source Data Main.xlsx" \
  --supplement "v2_Hayes Source Data Supplement.xlsx" \
  --features "structural_features.csv" \
  --outdir "egfr_results" 

## Step 10 — Inspect the results

In [ ]:
import json
import pandas as pd

with open("egfr_results/metrics.json") as f:
    metrics = json.load(f)

importance = pd.read_csv("egfr_results/structural_feature_importance.csv")
training_table = pd.read_csv("egfr_results/model_training_table.csv")

print("Validation metrics:")
print(json.dumps(metrics, indent=2))

print("\nMost predictive structural features:")
display(importance)

print("\nTraining data:")
display(training_table.head(10))

## Step 11 — Download the model outputs

In [ ]:
from google.colab import files

files.download("egfr_results/metrics.json")
files.download("egfr_results/structural_feature_importance.csv")
files.download("egfr_results/model_training_table.csv")

## Optional — Predict a new mutation–drug pair

Upload a completed `new_predictions.csv` containing the mutation, drug, and the same structural columns used for training. Then rerun the command with:

`--predict new_predictions.csv`

The output will be saved as `egfr_results/predictions.csv`.

In [ ]:
# Upload new_predictions.csv first, then run:
#
# !python egfr_resistance_pipeline.py \
#   --main "v2_Hayes Source Data Main.xlsx" \
#   --supplement "v2_Hayes Source Data Supplement.xlsx" \
#   --features "structural_features.csv" \
#   --predict "new_predictions.csv" \
#   --outdir "egfr_results"
#
# files.download("egfr_results/predictions.csv")